# Sandbox: Yair

Conflict, Democracy and Arms project. This is your personal notebook for trying things out.
Only Yair edits this file, so it never causes a merge conflict with Nicolas.

**How to use it**

1. Pull first (GitHub Desktop: Fetch origin, then Pull).
2. Work through the tasks in order. Every step has its own empty cell. Write your own code there.
3. Open a hint only when you are stuck. Hint 1 is a small nudge, later hints say more.
4. Run the checkpoint after each task. OK means done.
5. When a task is done, paste the working code into the main notebook listed below and push.

| Part | Task | Paste into |
| --- | --- | --- |
| Practice | How a merge works, on toy data | nothing, just for learning |
| A | SIPRI: arms imports per country-year and top suppliers | `04_sipri.ipynb` |
| B | Descriptives: top importers, imports by regime, democracy in war and peace | `06_analysis.ipynb`, Part A |
| C | Chart functions: arms line, suppliers, world map, democracy line, main chart | `07_charts.ipynb`, then `app/panels_arms.py` (C1 to C3) and `app/panels_analysis.py` (C4, C5) |

Variable, column and file names come from `CONVENTIONS.md`. Use them exactly, otherwise the merge in 05 breaks.

## How we write code in this project

We want code we understand completely, so we write it slowly and in small pieces. More lines are fine.

1. **One operation per line.** No long chains like `df.groupby(...).agg(...).reset_index().rename(...)`. Do each step on its own line and give the result a name.
2. **Look after every step.** Print `.shape`, `.head()` or a few values after each line that changes the data. If a number looks strange, stop and find out why.
3. **Simple tools first.** Filtering with conditions, `rename`, `sort_values`, dictionaries, lists, `for` loops and `if` statements. A loop that is slow but clear is better than one line we cannot explain.
4. **Comment in your own words.** Above each block, write one comment that says what it does.
5. **Use a new function only after trying it on a tiny example.** For example, before merging two big tables, merge two tables with three rows each (see the practice section).
6. **Use AI only for hints and explanations**, never to write the solution. Ask: "explain this error" or "what does this function do", and say that you want simple step-by-step code.

Tools we use: `read_csv`, `read_excel`, `head`, `shape`, `columns`, `isna`, `sum`, `rename`, `sort_values`, conditions with `&` and `|`, `isin`, `str.endswith`, `str.split`, `astype`, `for` loops, `iterrows`, dictionaries, lists, `pd.DataFrame(list_of_dicts)`, `unique`, `value_counts`, `between`, `notna`, `fillna`, `groupby` with one function at a time, `reset_index`, `nunique`, `merge` (after the practice), `concat`, `.copy()`, `to_csv`, and plotly express.

If you see a `SettingWithCopyWarning`, add `.copy()` after the filter that made the table, for example `df = panel[panel["iso3"] == "UKR"].copy()`.

TestYairPushPull:

1224535

## Setup

This cell only loads the packages and sets the folder paths. Run it every time you open the notebook.

In [ ]:
import pandas as pd
import plotly.express as px

RAW = "../data/raw/"
CLEAN = "../data/clean/"

pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__)

import os

---
## Practice: how a merge works (do this before any real merge)

A merge puts two tables side by side, matching rows that have the same value in a key column. We never did this in class, so we try it on tiny tables first, where we can check every row by eye.

Run the cell below. It creates two small tables.

In [ ]:
# Toy data: three countries with a democracy score, and arms imports for some of them
demo = pd.DataFrame({
    "iso3": ["CHE", "IND", "UKR"],
    "year": [2000, 2000, 2000],
    "democracy_index": [0.90, 0.70, 0.45],
})
arms = pd.DataFrame({
    "iso3": ["IND", "UKR", "SAU"],
    "year": [2000, 2000, 2000],
    "arms_imports_tiv": [900, 20, 1500],
})
print(demo)
print(arms)

**Step 1. Predict first.** Before you run anything, write in a comment: which countries appear in both tables? Which only in one?

In [ ]:
# your prediction here

**Step 2. Inner merge.** Merge `demo` and `arms` on `iso3` and `year` with `how="inner"`. Call the result `m_inner`. Print it. How many rows? Which country is missing and why?

In [ ]:
# your code here

**Step 3. Left merge.** Same, with `how="left"` and `demo` on the left. Call it `m_left`. Print it. What appears in `arms_imports_tiv` for CHE? Why?

In [ ]:
# your code here

**Step 4. Fill the gap.** In our project, a country with no arms deliveries imported nothing. Replace the missing value in `m_left` with 0. Print again. Also print `m_left.dtypes`: why is `arms_imports_tiv` now a decimal number (float)?

In [ ]:
# your code here

**Step 5. The danger: duplicate keys.** Run the cell below. It adds IND a second time to `arms`. Merge `demo` with `arms_dup` (left merge), call the result `m_dup` and count the rows. What happened to IND?

In [ ]:
arms_dup = pd.concat([arms, pd.DataFrame({"iso3": ["IND"], "year": [2000], "arms_imports_tiv": [5]})])
print(arms_dup)

In [ ]:
# your code here

**Step 6. Write the two rules in your own words** (as a comment): what does `how="left"` keep, and what must you check about the right table before every merge?

In [ ]:
# your rules here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

The function is `pd.merge(left_table, right_table, on=[...], how=...)`. `on` is a list of the key columns, `how` is the kind of merge as text.

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

`pd.merge(demo, arms, on=["iso3", "year"], how="inner")`. For the fill: `m_left["arms_imports_tiv"] = m_left["arms_imports_tiv"].fillna(0)`.
Rule of thumb for every real merge: print the number of rows before and after. With a left merge, the number must stay the same. If it grows, the right table has duplicate keys.

</details>

**Checkpoint.**

In [ ]:
assert len(m_inner) == 2, "inner keeps only countries in both tables"
assert len(m_left) == 3, "left keeps every row of demo"
assert sorted(m_left["iso3"]) == ["CHE", "IND", "UKR"], "left merge keeps the rows of the LEFT table (demo)"
assert m_left["arms_imports_tiv"].isna().sum() == 0, "fill the missing value with 0"
assert m_left.loc[m_left["iso3"] == "CHE", "arms_imports_tiv"].sum() == 0
assert len(m_dup) == 4, "the duplicate IND row in arms_dup creates an extra row"
print("OK")

---
## Part A: SIPRI (04)

Needs `crosswalk_sipri.csv` from 01 Crosswalk for A4.

**Before you start:** open `trade-register.csv` in a text editor (not Excel) and look at the first 15 lines. On which line do the column names start? Write it down.

### A1 Load the trade register

**Goal:** `sipri_raw` loaded correctly.

**Step 1.** Load it with `pd.read_csv`. Try first **without** extra arguments and read the error message, so you recognise it later. Then comment that line out with `#` (otherwise Restart and Run All stops here).

In [ ]:
# your code here

**Step 2.** Load it again with two extra arguments that skip the note lines and set the file's encoding. Print shape, column names and the first rows.

In [ ]:
# your code here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

You need `skiprows` (how many lines to skip before the column names) and `encoding` (the file uses Latin-1, not the usual UTF-8).

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

`pd.read_csv(RAW + "trade-register.csv", skiprows=11, encoding="latin-1")`

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert sipri_raw.shape == (62009, 17), f"expected (62009, 17), got {sipri_raw.shape}"
for col in ["Supplier", "Recipient", "Delivery year", "TIV delivery values"]:
    assert col in sipri_raw.columns, f"missing column {col}"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A2 Rename and keep what you need

**Goal:** `sipri` with exactly the columns `supplier`, `recipient`, `year`, `tiv`, `category`, `status`.

**Step 1.** Write a dictionary `new_names`: `Supplier` → `supplier`, `Recipient` → `recipient`, `Delivery year` → `year`, `TIV delivery values` → `tiv`, `Armament category` → `category`, `Status` → `status`. Rename with `rename(columns=new_names)` and call the result `sipri_raw_renamed`. Without `columns=` nothing gets renamed and you get no error, so print the columns afterwards.

In [ ]:
# your code here

**Step 2.** Make a list `keep` with the six new names, in this order, and select only those columns: `sipri = sipri_raw_renamed[keep].copy()`. Print the first rows.

In [ ]:
# your code here

**Step 3.** Some names may have spaces at the end, which breaks the matching in A4. Remove them: `sipri["recipient"] = sipri["recipient"].str.strip()`. 01 Crosswalk does the same with the names.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(sipri.columns) == ["supplier", "recipient", "year", "tiv", "category", "status"]
assert len(sipri) == 62009, "no rows should be lost in this step"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A3 Drop undelivered deals and non-states

**Goal:** only delivered arms to states. Write down after each step how many rows were removed.

**Step 1.** Count rows with a missing `year` (`sipri["year"].isna().sum()`). It should be 1,220: deals ordered but not delivered by 2025. Keep only rows where `year` is not missing. Print the row count before and after.

In [ ]:
# your code here

**Step 2.** Convert `year` to whole numbers with `astype(int)`.

In [ ]:
# your code here

**Step 3.** Look at the recipient names that end with a star: filter with `sipri["recipient"].str.endswith("*")` and print the unique names. One star = non-state actor, two stars = organisation. `endswith("*")` catches both, because a name ending in `**` also ends in `*`. Then keep only rows that do **not** end with a star (the `~` sign means "not").

In [ ]:
# your code here

**Step 4.** Print the names that contain "unknown" (`str.contains("unknown")`) to see the exact spelling. Then keep only rows where `recipient` is not equal to it. Print the final row count and write all removed numbers in a comment.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

Each filter on its own line, with the number printed before and after: `before = len(sipri)`, filter, `print(before - len(sipri), "rows removed")`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert sipri["year"].dtype.kind == "i"
assert not sipri["recipient"].str.endswith("*").any()
assert sipri["recipient"].str.contains("unknown").sum() == 0
assert 50000 < len(sipri) <= 62009 - 1220, "too many or too few rows removed"
print("rows left:", len(sipri))
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A4 Add iso3 with a merge

**Goal:** every remaining row has an `iso3`; rows that cannot be matched are removed on purpose and written down.

**Step 1.** Load `crosswalk_sipri.csv`. Print its first rows. The key is `recipient`.

In [ ]:
# your code here

**Step 2.** Check that every name appears only once in the crosswalk: `crosswalk_sipri["recipient"].duplicated().sum()` must be 0. Save `rows_before_merge = len(sipri)`.

In [ ]:
# your code here

**Step 3.** Merge with `how="left"`, `sipri` on the left, on `recipient`, and overwrite `sipri`. Print the length again. It must equal `rows_before_merge`.

In [ ]:
# your code here

**Step 4.** List the recipients without `iso3`: filter the rows where `iso3` is missing and print the unique names. Most should be fixed in the crosswalk (01): historical states map to the V-Dem unit of those years.

In [ ]:
# your code here

**Step 5.** Names that really have no V-Dem country: print how many rows and how much TIV they have, write that in the decisions table, then keep only rows where `iso3` is not missing.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

Exactly like the practice: `pd.merge(sipri, crosswalk_sipri, on="recipient", how="left")`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert "iso3" in sipri.columns
assert sipri["iso3"].isna().sum() == 0, "drop the remaining unmatched rows in step 5 (and write them down)"
assert len(sipri) <= rows_before_merge, "the merge added rows: a name appears twice in crosswalk_sipri"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A5 Imports per country-year

**Goal:** `sipri_cy` with exactly the columns `iso3`, `year`, `arms_imports_tiv`.

**Step 1.** Check one case by hand: filter India in 2000, print the rows, and add up `tiv` with `.sum()`. Store the number in `india_2000_hand`.

In [ ]:
# your code here

**Step 2.** Now for all, one line each: group by `["iso3", "year"]` and take `["tiv"].sum()`; then `reset_index()`; then rename `tiv` to `arms_imports_tiv`. Call the result `sipri_cy`.

In [ ]:
# your code here

**Step 3.** Compare: does `sipri_cy` give the same number for India 2000 as your hand calculation?

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

A year without deliveries does not appear in `sipri_cy` at all. That is fine; it becomes 0 in the merge (05), like in the practice.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(sipri_cy.columns) == ["iso3", "year", "arms_imports_tiv"]
assert sipri_cy.duplicated(["iso3", "year"]).sum() == 0
assert (sipri_cy["arms_imports_tiv"] >= 0).all()
india = sipri_cy[(sipri_cy["iso3"] == "IND") & (sipri_cy["year"] == 2000)]["arms_imports_tiv"].sum()
assert abs(india - india_2000_hand) < 0.001, "India 2000 differs from your hand calculation"
assert abs(sipri_cy["arms_imports_tiv"].sum() - sipri["tiv"].sum()) < 0.01, "the total TIV must not change"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A6 Top suppliers and save

**Goal:** `sipri_suppliers` with exactly the columns `iso3`, `year`, `supplier`, `tiv`, where only the top 3 suppliers of each country keep their name and the rest is called "Other".

**Step 1.** One country first, one line each: filter India; group by `supplier` and take `["tiv"].sum()`; `reset_index()`; `sort_values("tiv", ascending=False)`; `head(3)`, and call this last result `top`. Print it. These are India's top 3. Turn the names into a list with `list(top["supplier"])`.

In [ ]:
# your code here

**Step 2.** Store them for all countries: `top3 = {}` (an empty dictionary). Loop over `sipri["iso3"].unique()`. For each country, do what you did for India and save the list: `top3[code] = ...`. Print `top3["IND"]`.

In [ ]:
# your code here

**Step 3.** New column with a loop: an empty list `names`. Loop over `sipri` with `iterrows()`. If `row["supplier"]` is in `top3[row["iso3"]]`, append the supplier name, otherwise append `"Other"`. Then `sipri["supplier_group"] = names`.

In [ ]:
# your code here

**Step 4.** One line each: group by `["iso3", "year", "supplier_group"]` and take `["tiv"].sum()`; `reset_index()`; rename `supplier_group` to `supplier`. Call it `sipri_suppliers`.

In [ ]:
# your code here

**Step 5.** Save `sipri_cy.csv` and `sipri_suppliers.csv` with `index=False`.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

`if row["supplier"] in top3[row["iso3"]]:` checks whether the name is in that country's list. The loop over 60,000 rows takes a few seconds. That is fine.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(sipri_suppliers.columns) == ["iso3", "year", "supplier", "tiv"]
assert sipri_suppliers.duplicated(["iso3", "year", "supplier"]).sum() == 0
assert sipri_suppliers.groupby("iso3")["supplier"].nunique().max() <= 4, "top 3 + Other"
assert set(sipri_suppliers[sipri_suppliers["iso3"] == "IND"]["supplier"]) <= set(top3["IND"] + ["Other"])
assert abs(sipri_suppliers["tiv"].sum() - sipri_cy["arms_imports_tiv"].sum()) < 0.01, "the total TIV must not change"
assert os.path.exists(CLEAN + "sipri_cy.csv") and os.path.exists(CLEAN + "sipri_suppliers.csv")
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

---
## Part B: Descriptives (06 Part A)

Needs `panel.csv` from 05 Merge.

### B1 Top 15 importers

**Goal:** `top15`: the 15 countries with the highest total TIV since 1950.

**Step 1.** Load `panel.csv` into `panel`.

In [ ]:
# your code here

**Step 2.** Group by `country_name`, sum `arms_imports_tiv`. Call it `totals`.

In [ ]:
# your code here

**Step 3.** Sort from large to small and keep the first 15 with `.head(15)`. Call it `top15` and print it. Any surprises? Write one sentence.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert isinstance(top15, pd.Series), "keep top15 as the result of groupby + sum (a Series), without reset_index"
assert len(top15) == 15
assert list(top15) == sorted(top15, reverse=True), "sort from large to small"
assert top15.iloc[0] == totals.max()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### B2 Imports by regime over time

**Goal:** `by_regime`: total arms imports per year for each regime label.

**Step 1.** One line each: group by `["year", "regime_label"]`, sum `arms_imports_tiv`, reset the index. Call it `by_regime`. Print the rows for 1990.

In [ ]:
# your code here

**Step 2.** Plot it with `px.line(by_regime, x="year", y="arms_imports_tiv", color="regime_label")`. Write one sentence on what you see.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert set(by_regime.columns) == {"year", "regime_label", "arms_imports_tiv"}
assert abs(by_regime["arms_imports_tiv"].sum() - panel["arms_imports_tiv"].sum()) < 0.01
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### B3 Democracy in war and peace

**Goal:** average democracy index in conflict years versus peace years.

**Step 1.** Make two filtered tables `conflict_years` and `peace_years`, take the mean of `democracy_index` for each. Print both.

In [ ]:
# your code here

**Step 2.** Repeat only for wars (`max_intensity == 2`) versus peace. Is the gap larger? Write one sentence.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

Filter first, then take the mean. Give each filtered table a name.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert (conflict_years["conflict_active"] == 1).all()
assert (peace_years["conflict_active"] == 0).all()
assert len(conflict_years) + len(peace_years) == len(panel)
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

---
## Part C: Chart functions (07)

Each function takes a dataframe and returns a plotly figure. Test it here, then copy it unchanged into `07_charts.ipynb` and from there into `app/panels_arms.py` (C1 to C3) or `app/panels_analysis.py` (C4, C5). Both files need `import pandas as pd` and `import plotly.express as px` at the top.

**Rules for every timeline:** x-axis 1950 to 2025, height 180, margins `dict(l=60, r=10, t=30, b=10)` with `fig.update_yaxes(automargin=False)`, the legend above the chart, a short chart title, a y-axis title with the unit, no x-axis title, and a dashed vertical line at `year` if one is passed. Nicolas uses the same settings, so all timelines line up in the app (see `CONVENTIONS.md`).

Build each function in two rounds: first write the code **without** a function for one country and look at the chart. Only when the chart is right, put the same lines into a `def`, replace the country code with the argument `iso3`, and use only the arguments inside (never a global table).

### C0 Load the clean files

**Goal:** the chart data loaded from the CSV files, exactly as the app will do it.

**Step 1.** Load `panel.csv` and `sipri_suppliers.csv` from `CLEAN`. Agree with Nicolas on the colour dictionaries (he writes them in his D0).

In [ ]:
# your code here

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### C1 arms_line(panel, iso3, year=None)

**Goal:** arms imports over time for one country.

**Step 1.** Without a function: filter `panel` to `"IND"`, draw `px.bar` with `x="year"`, `y="arms_imports_tiv"` and `labels={"arms_imports_tiv": "Arms imports, TIV millions"}`.

In [ ]:
# your code here

**Step 2.** Same look as all timelines: `fig.update_xaxes(range=[1950, 2025])`, `fig.update_layout(height=180, margin=dict(l=60, r=10, t=30, b=10))`, `fig.update_yaxes(automargin=False)`.

In [ ]:
# your code here

**Step 3.** If `year` is given, add the dashed line (`if year is not None:` then `fig.add_vline(x=year, line_dash="dash")`).

In [ ]:
# your code here

**Step 4.** Wrap it in `def arms_line(panel, iso3, year=None):` with `return fig` at the end.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
fig = arms_line(panel, "IND", 1999)
assert list(fig.layout.xaxis.range) == [1950, 2025]
only_ukr = panel[panel["iso3"] == "UKR"]
assert sum(len(t.x) for t in arms_line(only_ukr, "IND").data) == 0, "filter the argument panel, not the global table"
assert len(fig.layout.shapes) == 1
assert len(arms_line(panel, "IND").layout.shapes) == 0
fig.show()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### C2 supplier_area(suppliers, iso3, year=None)

**Goal:** top 3 suppliers + Other, stacked bars by year.

**Step 1.** Filter the argument `suppliers` (not the global `sipri_suppliers`) to one country and draw `px.bar` with `x="year"`, `y="tiv"`, `color="supplier"`. Bars stack by default and show years without deliveries honestly as gaps.

In [ ]:
# your code here

**Step 2.** Same x-axis range, height, margins, legend above, and the dashed year line as C1.

In [ ]:
# your code here

**Step 3.** A country with no imports: after filtering, `if len(df) == 0:` return an empty figure (`px.bar()`) with `fig.add_annotation(text="No arms imports recorded", showarrow=False)` and the same height.

In [ ]:
# your code here

**Step 4.** Wrap it in the function.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
only_ind = sipri_suppliers[sipri_suppliers["iso3"] == "IND"]
empty = supplier_area(only_ind, "SAU")
assert len(empty.layout.annotations) == 1, "filter the argument, and show a note when there is no data"
supplier_area(sipri_suppliers, "IND", year=1999).show()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### C3 world_map(panel, year, column)

**Goal:** choropleth of `democracy_index` or `arms_imports_tiv` for one year.

**Step 1.** Filter `panel` to one year (say 2000) with `.copy()`. Draw `px.choropleth(df, locations="iso3", color="democracy_index", hover_name="country_name")`.

In [ ]:
# your code here

**Step 2.** Try `arms_imports_tiv`. A few huge importers make every other country look the same colour. Simple fix: with one `if` / `else` on `column`, pass `range_color=[0, 2000]` for arms (values above 2000 get the darkest colour, hover still shows the true value) and `range_color=[0, 1]` for democracy. Set the colour bar title with `labels={column: title}`.

In [ ]:
# your code here

**Step 3.** Wrap it in `def world_map(panel, year, column):`, using `color=column`.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

A few V-Dem codes are not official ISO codes (for example XKX Kosovo, PSG Gaza, SML Somaliland), so plotly cannot draw them on the map. That is fine; mention it in the report.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
before = panel.copy()
world_map(panel, 2000, "democracy_index").show()
world_map(panel, 2000, "arms_imports_tiv").show()
assert panel.equals(before), "the function must not change panel (use .copy() after filtering)"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### C4 democracy_line(panel, iso3, year=None)

**Goal:** democracy index over time for one country.

**Step 1.** Filter `panel` to one country, `px.line` with `x="year"`, `y="democracy_index"`.

In [ ]:
# your code here

**Step 2.** Fix the y-axis from 0 to 1 (`fig.update_yaxes(range=[0, 1])`), same x-axis, height and margins as C1, the dashed year line, then wrap it in the function.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
fig = democracy_line(panel, "UKR", year=2014)
assert list(fig.layout.xaxis.range) == [1950, 2025]
only_ind = panel[panel["iso3"] == "IND"]
assert sum(len(t.x) for t in democracy_line(only_ind, "UKR").data) == 0, "filter the argument panel, not the global table"
assert len(fig.layout.shapes) == 1
assert len(democracy_line(panel, "UKR").layout.shapes) == 0
fig.show()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### C5 main_chart(event_time)

**Goal:** the main result: `dem_change` by `t`, one line per importer group.

**Step 1.** Load `event_time.csv` (made together in 06 Part B). Print it. You should see one row per `t` and importer group.

In [ ]:
# your code here

**Step 2.** `px.line` with `x="t"`, `y="dem_change"`, `color="importer_group"`.

In [ ]:
# your code here

**Step 3.** Add a vertical line at 0 (`add_vline`), a horizontal line at 0 (`add_hline`), and axis titles "Years since conflict onset" and "Change in democracy index" (`fig.update_layout(xaxis_title=..., yaxis_title=...)`). Wrap it in the function.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
event_time = pd.read_csv(CLEAN + "event_time.csv")
fig = main_chart(event_time)
assert len(fig.data) == 2, "one line per importer group"
assert len(fig.layout.shapes) == 2, "one vertical and one horizontal line"
fig.show()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

## Before you move code into the main notebook

- [ ] Kernel → Restart and Run All runs this sandbox from top to bottom without an error
- [ ] All checkpoints above print OK
- [ ] Names follow `CONVENTIONS.md`
- [ ] You wrote the "in your own words" sentence for every task
- [ ] You can explain every line to your partner without looking at the hints
- [ ] Copy only the working cells into the numbered main notebook (no experiments, no long printouts)
- [ ] Run the main notebook with Restart and Run All, then commit and push